# Crossover EdTech Toolkit: Full Analysis Pipeline (Python)

This notebook runs the **complete analysis pipeline** for a 2x2 crossover study
evaluating the impact of generative AI on student learning outcomes.

Each section corresponds to one of the modular scripts (`00` through `09`).
You can run the notebook top-to-bottom, or jump to any section after running
Sections 0 and 1 (setup and data import).

---

## 0. Setup

Import packages, set paths, and configure plot defaults.

In [ ]:
import sys, os, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import pingouin as pg
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

print(f"Python {sys.version}")
for pkg_name in ["numpy", "pandas", "scipy", "statsmodels", "pingouin",
                 "matplotlib", "seaborn"]:
    mod = importlib.import_module(pkg_name)
    print(f"  {pkg_name:20s} {getattr(mod, '__version__', '?')}")

In [ ]:
# Paths (adjust PROJECT_ROOT if running from a different location)
PROJECT_ROOT = Path(".").resolve().parent.parent
DATA_DIR     = PROJECT_ROOT / "sample_data"
OUTPUT_DIR   = PROJECT_ROOT / "output"
TABLES_DIR   = OUTPUT_DIR / "tables"
FIGURES_DIR  = OUTPUT_DIR / "figures"
MODELS_DIR   = OUTPUT_DIR / "models"

for d in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")

In [ ]:
# Reproducibility & style
RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "font.family": "serif",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

PALETTE = {"noAI": "#E69F00", "AI": "#009E73"}
SEQ_PALETTE = {"AB": "#0072B2", "BA": "#D55E00"}
LIKERT_COLORS = {
    "Strongly Disagree": "#d73027", "Disagree": "#fc8d59",
    "Neutral": "#ffffbf", "Agree": "#91bfdb", "Strongly Agree": "#4575b4",
}

print("Setup complete.")

---
## 1. Data Import & Cleaning

Read the CSV, validate columns, encode factors, report missing data,
flag outliers, and create derived variables.

In [ ]:
data_file = DATA_DIR / "crossover_sample_data.csv"
df_raw = pd.read_csv(data_file)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns.")
df_raw.head()

In [ ]:
# Validate and encode factors
df = df_raw.copy()
df["participant_id"] = df["participant_id"].astype(str)
df["group"]     = pd.Categorical(df["group"], categories=["A", "B"])
df["sequence"]  = pd.Categorical(df["sequence"], categories=["AB", "BA"])
df["period"]    = pd.Categorical(
    df["period"].map({1: "Period 1", 2: "Period 2"}),
    categories=["Period 1", "Period 2"], ordered=True)
df["condition"] = pd.Categorical(df["condition"], categories=["noAI", "AI"])

# Missing data report
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
# Drop rows with missing primary outcome
df = df.dropna(subset=["score"]).reset_index(drop=True)

# Wide format for within-subject differences
score_wide = df.pivot_table(
    index=["participant_id", "sequence"],
    columns="condition", values=["score", "rubric_score"], aggfunc="first"
).reset_index()
score_wide.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c
                       for c in score_wide.columns]
score_wide["score_diff"] = score_wide["score_AI"] - score_wide["score_noAI"]
df_wide = score_wide.copy()

# Period sums for carryover test
df["period_num"] = df["period"].map({"Period 1": 1, "Period 2": 2}).astype(int)
df_period_sums = (
    df.groupby(["participant_id", "sequence"], observed=True)["score"]
    .sum().reset_index().rename(columns={"score": "score_sum"})
)

# Outlier flags (IQR)
def flag_outliers(g):
    q1, q3 = g["score"].quantile(0.25), g["score"].quantile(0.75)
    iqr = q3 - q1
    g["is_outlier"] = (g["score"] < q1 - 1.5*iqr) | (g["score"] > q3 + 1.5*iqr)
    return g

df = df.groupby("condition", group_keys=False, observed=True).apply(flag_outliers)
print(f"Outliers flagged: {df['is_outlier'].sum()}")
print(f"Wide-format participants: {len(df_wide)}")

---
## 2. Descriptive Statistics

In [ ]:
by_condition = (
    df.groupby("condition", observed=True)
    .agg(n=("score","size"), mean_score=("score","mean"),
         sd_score=("score","std"), median_score=("score","median"),
         mean_rubric=("rubric_score","mean"), sd_rubric=("rubric_score","std"),
         mean_time=("time_spent","mean"), sd_time=("time_spent","std"))
    .round(2).reset_index()
)
by_condition

In [ ]:
by_cond_period = (
    df.groupby(["condition","period"], observed=True)
    .agg(n=("score","size"), mean_score=("score","mean"), sd_score=("score","std"))
    .round(2).reset_index()
)
by_cond_period

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=df, x="condition", y="score", palette=PALETTE, ax=axes[0])
sns.stripplot(data=df, x="condition", y="score", color="black", alpha=0.15, ax=axes[0])
axes[0].set_title("Score by Condition")

sns.boxplot(data=df, x="period", y="score", hue="condition", palette=PALETTE, ax=axes[1])
axes[1].set_title("Score by Period & Condition")

axes[2].hist(df_wide["score_diff"].dropna(), bins=15, color="steelblue", edgecolor="white")
axes[2].axvline(0, color="red", linestyle="--")
axes[2].axvline(df_wide["score_diff"].mean(), color="darkblue")
axes[2].set_title("Within-Subject Differences (AI - noAI)")
axes[2].set_xlabel("Difference")

fig.tight_layout()
plt.show()

---
## 3. Instrument Validation

Cronbach's alpha, V de Aiken, ICC.

In [ ]:
likert_cols = [c for c in df.columns if c.startswith("likert_")]
likert_data = df[likert_cols].dropna()

alpha_val, ci = pg.cronbach_alpha(likert_data)
print(f"Cronbach's alpha = {alpha_val:.3f}  95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"Items: {len(likert_cols)}, Complete obs: {len(likert_data)}")

In [ ]:
# Item-total correlations and alpha-if-dropped
rows = []
for col in likert_cols:
    rest = [c for c in likert_cols if c != col]
    r_drop, _ = stats.pearsonr(likert_data[col], likert_data[rest].sum(axis=1))
    a_drop, _ = pg.cronbach_alpha(likert_data[rest]) if len(rest) >= 2 else (np.nan, None)
    rows.append({"item": col, "r_drop": round(r_drop, 3), "alpha_if_dropped": round(a_drop, 3)})

pd.DataFrame(rows)

---
## 4. Paired Comparisons

Paired t-test, Wilcoxon signed-rank, QQ plot.

In [ ]:
# Normality check
diffs = df_wide["score_diff"].dropna()
w_stat, w_p = stats.shapiro(diffs)
print(f"Shapiro-Wilk: W = {w_stat:.4f}, p = {w_p:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
stats.probplot(diffs, plot=ax)
ax.set_title("Q-Q Plot of Score Differences")
plt.show()

In [ ]:
# Paired t-test
t_res = pg.ttest(df_wide["score_AI"], df_wide["score_noAI"], paired=True)
print("Paired t-test (Score):")
t_res

In [ ]:
# Wilcoxon signed-rank
wilcox_res = pg.wilcoxon(df_wide["score_AI"], df_wide["score_noAI"])
print("Wilcoxon signed-rank (Score):")
wilcox_res

---
## 5. Mixed ANOVA & Linear Mixed Model

In [ ]:
# Mixed ANOVA
anova_res = pg.mixed_anova(
    data=df, dv="score", within="condition",
    between="sequence", subject="participant_id")
anova_res

In [ ]:
# Post-hoc
posthoc = pg.pairwise_tests(
    data=df, dv="score", within="condition",
    between="sequence", subject="participant_id", padjust="bonf")
posthoc

In [ ]:
# Linear Mixed Model
df_lmm = df.copy()
df_lmm["cond_num"] = (df_lmm["condition"] == "AI").astype(int)
df_lmm["period_num"] = df_lmm["period"].map({"Period 1": 1, "Period 2": 2}).astype(float)
df_lmm["seq_num"] = (df_lmm["sequence"] == "BA").astype(int)

lmm = smf.mixedlm("score ~ cond_num + period_num + seq_num",
                   data=df_lmm, groups=df_lmm["participant_id"]).fit(reml=True)
print(lmm.summary())

---
## 6. Effect Sizes

In [ ]:
def cohens_d_paired(x, y):
    d_arr = x - y
    n = len(d_arr)
    d = d_arr.mean() / d_arr.std(ddof=1)
    se = np.sqrt(1/n + d**2/(2*n))
    t_val = d * np.sqrt(n)
    ci_t = stats.t.interval(0.95, df=n-1, loc=t_val)
    return {"d": round(d, 3),
            "CI": [round(ci_t[0]/np.sqrt(n), 3), round(ci_t[1]/np.sqrt(n), 3)],
            "n": n}

d_score = cohens_d_paired(df_wide["score_AI"].dropna().values,
                          df_wide["score_noAI"].dropna().values)
print(f"Cohen's d (paired) = {d_score['d']}  CI = {d_score['CI']}")

# Hedges' g correction
J = 1 - 3 / (4*(d_score['n']-1) - 1)
g = round(d_score['d'] * J, 3)
print(f"Hedges' g = {g}")

---
## 7. Carryover Analysis

In [ ]:
sums_AB = df_period_sums.loc[df_period_sums["sequence"]=="AB", "score_sum"].values
sums_BA = df_period_sums.loc[df_period_sums["sequence"]=="BA", "score_sum"].values

t_carry, p_carry = stats.ttest_ind(sums_AB, sums_BA, equal_var=False)
u_carry, pu_carry = stats.mannwhitneyu(sums_AB, sums_BA, alternative="two-sided")

print(f"Grizzle's test: t = {t_carry:.3f}, p = {p_carry:.4f}")
print(f"Mann-Whitney U: U = {u_carry:.0f}, p = {pu_carry:.4f}")

if p_carry < 0.10:
    print("*** CARRYOVER DETECTED (alpha=0.10). Use Period 1 only. ***")
else:
    print("No carryover detected. Crossover analysis is valid.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(data=df_period_sums, x="sequence", y="score_sum",
            palette=SEQ_PALETTE, ax=ax, width=0.5)
ax.set_title(f"Carryover Test (p = {p_carry:.3f})")
ax.set_ylabel("Sum of scores (P1 + P2)")
plt.show()

---
## 8. Perception Analysis (Likert)

In [ ]:
likert_means = df.groupby("condition", observed=True)[likert_cols].mean().round(2)
likert_means

In [ ]:
# Wilcoxon per item with Holm correction
df_lik_wide = df[["participant_id", "condition"] + likert_cols].melt(
    id_vars=["participant_id", "condition"], var_name="item", value_name="resp"
).pivot_table(index=["participant_id","item"], columns="condition",
              values="resp", aggfunc="first").reset_index()
df_lik_wide.columns = ["participant_id", "item", "AI", "noAI"]

wilcox_rows = []
for col in likert_cols:
    sub = df_lik_wide[df_lik_wide["item"]==col].dropna(subset=["AI","noAI"])
    if len(sub) >= 10:
        w = pg.wilcoxon(sub["AI"], sub["noAI"])
        wilcox_rows.append({"item": col, "W": w["W-val"].values[0],
                            "p": w["p-val"].values[0], "RBC": w["RBC"].values[0]})

wdf = pd.DataFrame(wilcox_rows)
_, wdf["p_holm"], _, _ = multipletests(wdf["p"], method="holm")
wdf.round(4)

---
## 9. Publication Visualizations

In [ ]:
# Interaction plot
cell_stats = (
    df.groupby(["sequence","period","condition"], observed=True)
    .agg(mean_score=("score","mean"), se_score=("score","sem"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 5))
for seq, color in SEQ_PALETTE.items():
    sub = cell_stats[cell_stats["sequence"]==seq].sort_values("period")
    label = f"{seq} ({'noAI->AI' if seq=='AB' else 'AI->noAI'})"
    ax.errorbar([0,1], sub["mean_score"].values, yerr=1.96*sub["se_score"].values,
                marker="o", ms=8, lw=1.5, capsize=4, color=color, label=label)
ax.set_xticks([0,1]); ax.set_xticklabels(["Period 1","Period 2"])
ax.set_ylabel("Mean Score"); ax.set_title("Crossover Interaction Plot")
ax.legend()
plt.show()

In [ ]:
# Spaghetti plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax_i, seq in enumerate(["AB", "BA"]):
    sub = df[df["sequence"]==seq].copy()
    sub["px"] = sub["period"].map({"Period 1": 0, "Period 2": 1})
    for pid, g in sub.groupby("participant_id"):
        g = g.sort_values("px")
        axes[ax_i].plot(g["px"], g["score"], color="grey", alpha=0.2, lw=0.7)
    for cond, c in PALETTE.items():
        csub = sub[sub["condition"]==cond]
        axes[ax_i].scatter(csub["px"], csub["score"], color=c, alpha=0.4, s=12, label=cond)
    axes[ax_i].set_xticks([0,1]); axes[ax_i].set_xticklabels(["Period 1","Period 2"])
    axes[ax_i].set_title(f"Sequence {seq}", fontweight="bold")
    axes[ax_i].legend(fontsize=8)
axes[0].set_ylabel("Score")
fig.suptitle("Individual Trajectories", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Forest plot
es_file = TABLES_DIR / "effect_sizes.csv"
if es_file.exists():
    es = pd.read_csv(es_file).sort_values("Estimate")
    fig, ax = plt.subplots(figsize=(9, max(3, len(es)*0.7)))
    y = np.arange(len(es))
    ax.axvline(0, ls="--", color="grey", lw=0.8)
    ax.errorbar(es["Estimate"], y,
                xerr=[es["Estimate"]-es["CI_lower"], es["CI_upper"]-es["Estimate"]],
                fmt="o", color="#0072B2", ms=7, capsize=3)
    ax.set_yticks(y); ax.set_yticklabels(es["Comparison"], fontsize=9)
    ax.set_xlabel("Effect Size"); ax.set_title("Forest Plot")
    fig.tight_layout()
    plt.show()
else:
    print("Run effect sizes section first or run 06_effect_sizes.py.")

In [ ]:
# Violin + box plot
fig, ax = plt.subplots(figsize=(5, 5))
parts = ax.violinplot(
    [df.loc[df["condition"]==c, "score"].dropna().values for c in ["noAI","AI"]],
    positions=[0,1], showextrema=False)
for pc, cond in zip(parts["bodies"], ["noAI","AI"]):
    pc.set_facecolor(PALETTE[cond]); pc.set_alpha(0.4)
bp = ax.boxplot(
    [df.loc[df["condition"]==c, "score"].dropna().values for c in ["noAI","AI"]],
    positions=[0,1], widths=0.15, patch_artist=True, showfliers=False)
for p, cond in zip(bp["boxes"], ["noAI","AI"]):
    p.set_facecolor(PALETTE[cond]); p.set_alpha(0.8)
ax.set_xticks([0,1]); ax.set_xticklabels(["noAI","AI"])
ax.set_ylabel("Score"); ax.set_title("Score Distribution")
plt.show()

---

**End of pipeline.** All tables have been saved to `output/tables/` and
all figures to `output/figures/`.